In [6]:
import numpy as np
from numpy import linalg
import cvxopt
import cvxopt.solvers
cvxopt.solvers.options['show_progress'] = False
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

Support Vector Machine (SVM) classifier with kernel support.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `kernel` | `'linear' \| 'polynomial' \| 'gaussian'` | `'linear'` | Kernel function used by the classifier |
| `C` | `float` or `None` | `None` | Regularisation parameter. `None` or `0` implies a **hard-margin SVM** |
| `gamma` | `float` | `0.5` | Bandwidth parameter for the **Gaussian (RBF) kernel** |
| `degree` | `int` | `3` | Degree of the **polynomial kernel** |
| `coef0` | `float` | `1.0` | Bias term used in the **polynomial kernel** |

In [3]:
class SVM:
    def __init__(self, kernel='linear', C=None, gamma=0.5, degree=3, coef0=1.0):
        self.kernel = kernel
        self.C      = float(C) if C else None
        self.gamma  = gamma
        self.degree = degree
        self.coef0  = coef0

    # ------------------------------------------------------------------ #
    #  Kernels                                                           #
    # ------------------------------------------------------------------ #

    def _kernel(self, x, z):
        if self.kernel == 'linear':
            return np.dot(x, z)
        if self.kernel == 'polynomial':
            return (np.dot(x, z) + self.coef0) ** self.degree
        if self.kernel == 'gaussian':
            return np.exp(-self.gamma * linalg.norm(x - z) ** 2)
        raise ValueError(f"Unknown kernel: '{self.kernel}'")

    def _gram(self, X):
        n = len(X)
        K = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                K[i, j] = self._kernel(X[i], X[j])
        return K

    # ------------------------------------------------------------------ #
    #  Fit                                                               #
    # ------------------------------------------------------------------ #

    def fit(self, X, y):
        """Solve the dual QP and store support vectors."""
        n, n_features = X.shape
        y = y.astype(float)
        K = self._gram(X)

        # ---- QP matrices ----
        P = cvxopt.matrix(np.outer(y, y) * K)
        q = cvxopt.matrix(-np.ones(n))
        A = cvxopt.matrix(y, (1, n), 'd')
        b = cvxopt.matrix(0.0)

        if self.C is None:                          # hard margin
            G = cvxopt.matrix(-np.eye(n))
            h = cvxopt.matrix(np.zeros(n))
        else:                                       # soft margin
            G = cvxopt.matrix(np.vstack([-np.eye(n), np.eye(n)]))
            h = cvxopt.matrix(np.hstack([np.zeros(n), np.ones(n) * self.C]))

        sol = cvxopt.solvers.qp(P, q, G, h, A, b)
        alphas = np.ravel(sol['x'])

        # ---- Support vectors ----
        mask          = alphas > 1e-5
        self._alphas  = alphas[mask]
        self._sv_X    = X[mask]
        self._sv_y    = y[mask]
        self._sv_idx  = np.where(mask)[0]

        # ---- Bias ----
        self._b = np.mean(
            self._sv_y
            - np.sum(self._alphas * self._sv_y * K[self._sv_idx][:, mask], axis=1)
        )

        # ---- Weight vector (linear only) ----
        self._w = (
            (self._alphas * self._sv_y) @ self._sv_X
            if self.kernel == 'linear' else None
        )

        return self

    # ------------------------------------------------------------------ #
    #  Predict                                                           #
    # ------------------------------------------------------------------ #

    def _decision_function(self, X):
        if self._w is not None:                     # fast path for linear
            return X @ self._w + self._b

        scores = np.array([
            sum(a * y * self._kernel(x, sv)
                for a, y, sv in zip(self._alphas, self._sv_y, self._sv_X))
            for x in X
        ])
        return scores + self._b

    def predict(self, X):
        """Return class labels {-1, +1}."""
        return np.sign(self._decision_function(X))

    # ------------------------------------------------------------------ #
    #  Convenience                                                       #
    # ------------------------------------------------------------------ #

    def score(self, X, y):
        """Fraction of correctly classified samples."""
        return np.mean(self.predict(X) == y)

    def __repr__(self):
        return (f"SVM(kernel={self.kernel!r}, C={self.C}, "
                f"gamma={self.gamma}, degree={self.degree})")

In [9]:
X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    random_state=42
)

y = np.where(y == 0, -1, 1)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [12]:
svm = SVM(kernel='linear', C=1.0)
svm.fit(X_train, y_train)
print(svm.score(X_test, y_test))

0.95
